In [1]:
import torch
from torch import nn

In [2]:
class vaemodel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.mu = nn.Linear(256, 6)  # Reduced latent space size
        self.std = nn.Linear(256, 6)  # Reduced latent space size

    def forward(self, x):
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out

model = vaemodel1()

In [5]:
vae_dict = torch.load("vae_pre_trained_w.pt", weights_only=True)
print(vae_dict.keys())

# Remove keys starting with 'decoder' and 'encoder.0'
keys_to_remove = [key for key in vae_dict.keys() if key.startswith('decoder') or key.startswith('encoder.0')]
for key in keys_to_remove:
    del vae_dict[key]

# Rename keys encoder.1 -> encoder.0, encoder.2 -> encoder.1, encoder.3 -> encoder.2, ...
encoder_dict = {}
for key in vae_dict.keys():
    if key.startswith('encoder.'):
        parts = key.split('.')
        layer_num = int(parts[1]) - 1
        new_key = f'encoder.{layer_num}.' + '.'.join(parts[2:])
        encoder_dict[new_key] = vae_dict[key]
    else:
        encoder_dict[key] = vae_dict[key]

print(encoder_dict.keys())
torch.save(encoder_dict, "encoder_pre_trained_w.pt")


odict_keys(['encoder.0.weight', 'encoder.0.bias', 'encoder.0.running_mean', 'encoder.0.running_var', 'encoder.0.num_batches_tracked', 'encoder.1.weight', 'encoder.1.bias', 'encoder.2.weight', 'encoder.2.bias', 'encoder.2.running_mean', 'encoder.2.running_var', 'encoder.2.num_batches_tracked', 'encoder.4.weight', 'encoder.4.bias', 'encoder.5.weight', 'encoder.5.bias', 'encoder.5.running_mean', 'encoder.5.running_var', 'encoder.5.num_batches_tracked', 'encoder.7.weight', 'encoder.7.bias', 'encoder.8.weight', 'encoder.8.bias', 'encoder.8.running_mean', 'encoder.8.running_var', 'encoder.8.num_batches_tracked', 'encoder.10.weight', 'encoder.10.bias', 'encoder.11.weight', 'encoder.11.bias', 'encoder.11.running_mean', 'encoder.11.running_var', 'encoder.11.num_batches_tracked', 'encoder.13.weight', 'encoder.13.bias', 'encoder.14.weight', 'encoder.14.bias', 'encoder.14.running_mean', 'encoder.14.running_var', 'encoder.14.num_batches_tracked', 'mu.weight', 'mu.bias', 'std.weight', 'std.bias', 'd

In [6]:
model.load_state_dict(torch.load("encoder_pre_trained_w.pt", weights_only=True))

<All keys matched successfully>